In [ ]:
from typing import Literal, TypedDict
from langgraph.graph import END, START, StateGraph

# 1. STATE DEFINITION
class ClassroomApprovalState(TypedDict):
    student_name: str
    request_type: Literal[
        "toilet_break",
        "leave",
        "half_day",
        "sick_room",
        "general_request",
    ]
    reason: str
    teacher_brief: str
    human_decision: str
    final_status: str

# 2. NODES
def prepare_teacher_brief(state: ClassroomApprovalState) -> dict:
    """Student ki request ke hisaab se teacher ke liye structured brief banata hai."""
    req_type = state["request_type"]
    student = state["student_name"]
    reason = state["reason"]

    policy_notes = {
        "toilet_break": "Quick washroom break (recommended: 5 mins max, keep pass).",
        "leave": "Full-day/multi-day leave request (check attendance & parent note).",
        "half_day": "Half-day early departure (verify gate pass & guardian pickup).",
        "sick_room": "Medical/health issue (priority check: send to sick room/nurse).",
        "general_request": "Other classroom/school errand (check ongoing lecture impact).",
    }

    note = policy_notes.get(req_type, "Standard classroom request.")
    brief = (
        f"Student: {student} | Category: {req_type.replace('_', ' ').title()}\n"
        f"Reason: {reason}\n"
        f"System Note: {note}"
    )
    return {"teacher_brief": brief}


def teacher_human_approval(state: ClassroomApprovalState) -> dict:
    """Teacher se terminal par approval (yes/no) leta hai."""
    print("\n" + "-" * 50)
    print("TEACHER APPROVAL REQUIRED")
    print("-" * 50)
    print(state["teacher_brief"])

    while True:
        decision = input("\nTeacher decision (yes/no): ").strip().lower()
        if decision in {"yes", "y"}:
            return {"human_decision": "yes"}
        if decision in {"no", "n"}:
            return {"human_decision": "no"}
        print("Invalid input! Kripya 'yes' ya 'no' enter karein.")


def approved_node(state: ClassroomApprovalState) -> dict:
    """Approval milne par final status set karta hai."""
    action_map = {
        "toilet_break": f"Approved: {state['student_name']} can go to the toilet.",
        "leave": f"Approved: Leave granted for {state['student_name']}.",
        "half_day": f"Approved: Half-day exit pass issued for {state['student_name']}.",
        "sick_room": f"Approved: {state['student_name']} may visit the sick room.",
        "general_request": f"Approved: Request granted for {state['student_name']}.",
    }
    msg = action_map.get(
        state["request_type"],
        f"Approved: Request granted for {state['student_name']}.",
    )
    return {"final_status": msg}


def rejected_node(state: ClassroomApprovalState) -> dict:
    """Rejection par final status set karta hai."""
    return {
        "final_status": (
            f"Rejected: Teacher did not approve '{state['request_type']}' "
            f"for {state['student_name']}."
        )
    }

# 3. CONDITIONAL ROUTING
def route_teacher_decision(
    state: ClassroomApprovalState,
) -> Literal["approved", "rejected"]:
    if state["human_decision"] == "yes":
        return "approved"
    return "rejected"

# 4. BUILD & COMPILE GRAPH
builder = StateGraph(ClassroomApprovalState)

builder.add_node("prepare_teacher_brief", prepare_teacher_brief)
builder.add_node("teacher_human_approval", teacher_human_approval)
builder.add_node("approved", approved_node)
builder.add_node("rejected", rejected_node)

builder.add_edge(START, "prepare_teacher_brief")
builder.add_edge("prepare_teacher_brief", "teacher_human_approval")

builder.add_conditional_edges(
    "teacher_human_approval",
    route_teacher_decision,
    {
        "approved": "approved",
        "rejected": "rejected",
    },
)

builder.add_edge("approved", END)
builder.add_edge("rejected", END)

graph = builder.compile()

# 5. RUN 5 CLASSROOM REQUESTS
classroom_requests: list[ClassroomApprovalState] = [
    {
        "student_name": "Aarav",
        "request_type": "toilet_break",
        "reason": "Urgent washroom break during lecture",
        "teacher_brief": "",
        "human_decision": "",
        "final_status": "",
    },
    {
        "student_name": "Ananya",
        "request_type": "leave",
        "reason": "Family function out of town for 2 days",
        "teacher_brief": "",
        "human_decision": "",
        "final_status": "",
    },
    {
        "student_name": "Rohan",
        "request_type": "half_day",
        "reason": "Dental appointment at 12:30 PM",
        "teacher_brief": "",
        "human_decision": "",
        "final_status": "",
    },
    {
        "student_name": "Meera",
        "request_type": "sick_room",
        "reason": "Severe headache and dizziness",
        "teacher_brief": "",
        "human_decision": "",
        "final_status": "",
    },
    {
        "student_name": "Kabir",
        "request_type": "general_request",
        "reason": "Need to return a reference book to the library",
        "teacher_brief": "",
        "human_decision": "",
        "final_status": "",
    },
]

results = []
for req in classroom_requests:
    outcome = graph.invoke(req)
    results.append(outcome)

print("\n" + "=" * 55)
print("END OF CLASS SUMMARY")
print("=" * 55)
for res in results:
    print(f"- {res['student_name']} ({res['request_type']}): {res['final_status']}")


--------------------------------------------------
TEACHER APPROVAL REQUIRED
--------------------------------------------------
Student: Aarav | Category: Toilet Break
Reason: Urgent washroom break during lecture
System Note: Quick washroom break (recommended: 5 mins max, keep pass).



--------------------------------------------------
TEACHER APPROVAL REQUIRED
--------------------------------------------------
Student: Ananya | Category: Leave
Reason: Family function out of town for 2 days
System Note: Full-day/multi-day leave request (check attendance & parent note).

--------------------------------------------------
TEACHER APPROVAL REQUIRED
--------------------------------------------------
Student: Rohan | Category: Half Day
Reason: Dental appointment at 12:30 PM
System Note: Half-day early departure (verify gate pass & guardian pickup).

--------------------------------------------------
TEACHER APPROVAL REQUIRED
--------------------------------------------------
Student: Meera | Category: Sick Room
Reason: Severe headache and dizziness
System Note: Medical/health issue (priority check: send to sick room/nurse).

--------------------------------------------------
TEACHER APPROVAL REQUIRED
--------------------------------------------------
Student: Kabir | Cat